# Scoring de Riesgo de Crédito Dinámico

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Construir un modelo de scoring crediticio** con `SNOWFLAKE.ML.CLASSIFICATION`
2. **Generar datos sintéticos** de solicitudes de crédito con perfiles de riesgo
3. **Implementar explicabilidad** con `CORTEX.COMPLETE()` para justificar decisiones
4. **Calcular ratios financieros** como features predictivas
5. **Construir un dashboard** de decisiones crediticias con Streamlit

---

## Lo Que Construirás

Un **sistema de scoring crediticio dinámico** que evalúa solicitudes:
- Clasificación multi-nivel (APROBADO/CONDICIONADO/RECHAZADO)
- Features financieras: ratio deuda-ingreso, historial de pagos, estabilidad laboral
- Explicación IA de cada decisión con CORTEX.COMPLETE
- Dashboard de gestión de solicitudes

**Valor de Negocio**: Automatizar decisiones crediticias con transparencia y trazabilidad.

---

## Requisitos Técnicos

- **Cuenta Snowflake** con Cortex ML y Cortex AI habilitados
- **Aproximadamente 18 minutos** para completar
- **100% SQL** (excepto dashboard Streamlit)

---

## Funcionalidades Clave de Snowflake

- `SNOWFLAKE.ML.CLASSIFICATION` — Modelo predictivo de riesgo
- `CORTEX.COMPLETE()` — Generación de explicaciones en lenguaje natural
- `GENERATOR()` — Datos sintéticos de solicitudes
- `NTILE()` — Segmentación en bandas de riesgo
- `OBJECT_CONSTRUCT` — Predicción inline

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Scoring Crediticio con IA

**Objetivo**: Evaluar automáticamente el riesgo de cada solicitud de crédito con explicaciones transparentes.

### Métricas Clave

- **Tasa de aprobación**: % de solicitudes aprobadas
- **Tasa de morosidad**: % de créditos impagados
- **Ratio deuda-ingreso (DTI)**: Indicador principal de capacidad de pago
- **Score de riesgo**: Probabilidad de impago (0-100)

### ¿Por Qué ML para Scoring?

Los modelos tradicionales (scorecards) son estáticos. ML permite:
- Actualizar el score con nuevos datos en tiempo real
- Capturar relaciones no lineales entre variables
- Explicar decisiones para cumplimiento regulatorio (GDPR/EBA)

In [ ]:
-- Configurar entorno de scoring crediticio
CREATE DATABASE IF NOT EXISTS SCORING_CREDITO_DB;
CREATE SCHEMA IF NOT EXISTS SCORING_CREDITO_DB.PUBLIC;
USE SCHEMA SCORING_CREDITO_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema, CURRENT_WAREHOUSE() AS wh;

---

## Paso 2: Definir Estructura de Datos

### Tablas

1. **`SOLICITANTES`** — Perfiles demográficos y financieros
2. **`HISTORIAL_CREDITO`** — Créditos anteriores y comportamiento de pago
3. **`SOLICITUDES`** — Solicitudes actuales con importe y plazo

### Variables de Riesgo

- **Ratio Deuda-Ingreso (DTI)**: Deuda total / Ingreso mensual
- **Historial de pagos**: % de pagos a tiempo en créditos anteriores
- **Estabilidad laboral**: Años en empleo actual
- **Antigüedad bancaria**: Años como cliente del banco
- **Utilización de crédito**: % del límite de crédito usado

In [ ]:
-- Crear estructura de tablas
CREATE OR REPLACE TABLE SOLICITANTES (
    solicitante_id VARCHAR(10) PRIMARY KEY,
    edad INTEGER,
    ingresos_mensuales DECIMAL(10,2),
    gastos_mensuales DECIMAL(10,2),
    deuda_actual DECIMAL(12,2),
    anos_empleo_actual INTEGER,
    tipo_empleo VARCHAR(30),
    estado_civil VARCHAR(20),
    num_dependientes INTEGER,
    vivienda VARCHAR(20),
    antiguedad_bancaria INTEGER
);

CREATE OR REPLACE TABLE HISTORIAL_CREDITO (
    credito_id VARCHAR(10) PRIMARY KEY,
    solicitante_id VARCHAR(10),
    tipo_credito VARCHAR(30),
    importe_original DECIMAL(12,2),
    fecha_apertura DATE,
    estado VARCHAR(20),
    pagos_a_tiempo_pct FLOAT,
    impagos_30d INTEGER,
    impagos_90d INTEGER,
    FOREIGN KEY (solicitante_id) REFERENCES SOLICITANTES(solicitante_id)
);

CREATE OR REPLACE TABLE SOLICITUDES (
    solicitud_id VARCHAR(10) PRIMARY KEY,
    solicitante_id VARCHAR(10),
    fecha_solicitud DATE,
    importe_solicitado DECIMAL(12,2),
    plazo_meses INTEGER,
    finalidad VARCHAR(50),
    es_moroso BOOLEAN,
    FOREIGN KEY (solicitante_id) REFERENCES SOLICITANTES(solicitante_id)
);

SELECT 'Tablas creadas correctamente' AS status;

---

## Paso 3: Generar Datos Sintéticos

### Qué Vamos a Crear

- **5.000 solicitantes** con perfiles financieros variados
- **15.000 registros de historial** de créditos anteriores
- **5.000 solicitudes** con 20% de morosidad

### Distribución Realista

- Solicitantes morosos tienden a: DTI alto (>40%), pagos tardíos, empleo inestable
- Solicitantes solventes: DTI bajo (<30%), pagos puntuales, empleo estable
- Ingresos: 1.200€ — 8.000€ mensuales

In [ ]:
-- Generar 5.000 solicitantes
INSERT INTO SOLICITANTES
SELECT
    'SOL' || LPAD(SEQ4()::STRING, 5, '0'),
    UNIFORM(21, 70, RANDOM()),
    ROUND(UNIFORM(1200, 8000, RANDOM()), 2),
    ROUND(UNIFORM(400, 4000, RANDOM()), 2),
    ROUND(UNIFORM(0, 80000, RANDOM()), 2),
    UNIFORM(0, 25, RANDOM()),
    ARRAY_CONSTRUCT('Indefinido','Temporal','Autónomo','Funcionario','Desempleado')[UNIFORM(0,4,RANDOM())]::VARCHAR,
    ARRAY_CONSTRUCT('Soltero','Casado','Divorciado','Viudo')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    UNIFORM(0, 4, RANDOM()),
    ARRAY_CONSTRUCT('Propiedad','Hipoteca','Alquiler','Familiares')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    UNIFORM(0, 30, RANDOM())
FROM TABLE(GENERATOR(ROWCOUNT => 5000));

-- Generar historial crediticio
INSERT INTO HISTORIAL_CREDITO
SELECT
    'CRE' || LPAD(SEQ4()::STRING, 6, '0'),
    'SOL' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 5, '0'),
    ARRAY_CONSTRUCT('Hipoteca','Personal','Tarjeta','Auto','Consumo')[UNIFORM(0,4,RANDOM())]::VARCHAR,
    ROUND(UNIFORM(1000, 200000, RANDOM()), 2),
    DATEADD('day', -UNIFORM(30, 3650, RANDOM()), CURRENT_DATE()),
    ARRAY_CONSTRUCT('Activo','Cerrado','Refinanciado')[UNIFORM(0,2,RANDOM())]::VARCHAR,
    ROUND(UNIFORM(50, 100, RANDOM()), 1),
    UNIFORM(0, 5, RANDOM()),
    UNIFORM(0, 2, RANDOM())
FROM TABLE(GENERATOR(ROWCOUNT => 15000));

-- Generar solicitudes (20% morosas)
INSERT INTO SOLICITUDES
SELECT
    'REQ' || LPAD(SEQ4()::STRING, 5, '0'),
    'SOL' || LPAD(UNIFORM(1, 5000, RANDOM())::STRING, 5, '0'),
    DATEADD('day', -UNIFORM(0, 365, RANDOM()), CURRENT_DATE()),
    ROUND(UNIFORM(3000, 100000, RANDOM()), 2),
    ARRAY_CONSTRUCT(12, 24, 36, 48, 60, 84)[UNIFORM(0,5,RANDOM())]::INT,
    ARRAY_CONSTRUCT('Vivienda','Consumo','Vehículo','Negocio','Consolidación','Estudios')[UNIFORM(0,5,RANDOM())]::VARCHAR,
    CASE WHEN UNIFORM(0,100,RANDOM()) < 20 THEN TRUE ELSE FALSE END
FROM TABLE(GENERATOR(ROWCOUNT => 5000));

SELECT 'Datos generados' AS status, 
    (SELECT COUNT(*) FROM SOLICITANTES) AS solicitantes,
    (SELECT COUNT(*) FROM HISTORIAL_CREDITO) AS historiales,
    (SELECT COUNT(*) FROM SOLICITUDES) AS solicitudes;

---

## Paso 4: Ingeniería de Variables Crediticias

### Features Financieras

| Variable | Descripción | Importancia |
|---|---|---|
| `dti_ratio` | Deuda total / Ingreso mensual | Alta — indica capacidad de pago |
| `pago_medio_historico` | % medio de pagos a tiempo | Alta — indica comportamiento |
| `impagos_totales` | Nº de impagos >30d | Alta — señal directa de riesgo |
| `estabilidad_empleo` | Años en empleo actual | Media — estabilidad económica |
| `utilizacion_credito` | Deuda / Límite de crédito | Media — apalancamiento |

In [ ]:
-- Crear features de scoring crediticio
CREATE OR REPLACE TABLE FEATURES_CREDITO AS
WITH hist_agg AS (
    SELECT
        solicitante_id,
        COUNT(*) AS num_creditos_previos,
        AVG(pagos_a_tiempo_pct) AS pago_medio_historico,
        SUM(impagos_30d) AS total_impagos_30d,
        SUM(impagos_90d) AS total_impagos_90d,
        SUM(CASE WHEN estado = 'Activo' THEN importe_original ELSE 0 END) AS deuda_activa
    FROM HISTORIAL_CREDITO GROUP BY 1
)
SELECT
    sol.solicitud_id,
    s.edad::FLOAT AS edad,
    (s.deuda_actual / NULLIF(s.ingresos_mensuales, 0))::FLOAT AS dti_ratio,
    s.ingresos_mensuales::FLOAT,
    s.gastos_mensuales::FLOAT,
    s.anos_empleo_actual::FLOAT AS estabilidad_empleo,
    CASE WHEN s.tipo_empleo = 'Indefinido' THEN 1 WHEN s.tipo_empleo = 'Funcionario' THEN 1 ELSE 0 END::FLOAT AS empleo_estable,
    s.num_dependientes::FLOAT,
    CASE WHEN s.vivienda = 'Propiedad' THEN 1 ELSE 0 END::FLOAT AS es_propietario,
    s.antiguedad_bancaria::FLOAT,
    COALESCE(h.num_creditos_previos, 0)::FLOAT AS num_creditos_previos,
    COALESCE(h.pago_medio_historico, 50)::FLOAT AS pago_medio_historico,
    COALESCE(h.total_impagos_30d, 0)::FLOAT AS total_impagos_30d,
    COALESCE(h.total_impagos_90d, 0)::FLOAT AS total_impagos_90d,
    sol.importe_solicitado::FLOAT,
    sol.plazo_meses::FLOAT,
    sol.es_moroso
FROM SOLICITUDES sol
JOIN SOLICITANTES s ON sol.solicitante_id = s.solicitante_id
LEFT JOIN hist_agg h ON sol.solicitante_id = h.solicitante_id;

SELECT * FROM FEATURES_CREDITO LIMIT 10;

---

## Paso 5: Entrenar Modelo de Scoring

### AutoML de Snowflake

El modelo evaluará automáticamente múltiples algoritmos y seleccionará el mejor para predecir morosidad.

In [ ]:
-- Dividir datos
CREATE OR REPLACE TABLE TRAIN_CREDITO AS SELECT * FROM FEATURES_CREDITO SAMPLE (80);
CREATE OR REPLACE TABLE TEST_CREDITO AS SELECT * FROM FEATURES_CREDITO WHERE solicitud_id NOT IN (SELECT solicitud_id FROM TRAIN_CREDITO);

-- Entrenar modelo
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION scoring_credito(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_CREDITO'),
    TARGET_COLNAME => 'ES_MOROSO'
);

---

## Paso 6: Evaluar y Puntuar

### Métricas de Rendimiento

Para scoring crediticio:
- **Precision** alta = menos aprobaciones erróneas a morosos
- **Recall** alto = detectar la mayoría de los morosos potenciales
- **AUC** > 0.80 es un buen modelo de riesgo

In [ ]:
-- Evaluar modelo
CALL scoring_credito!SHOW_EVALUATION_METRICS();
CALL scoring_credito!SHOW_CONFUSION_MATRIX();
CALL scoring_credito!SHOW_FEATURE_IMPORTANCE();

In [ ]:
-- Puntuar solicitudes con decisión y explicación IA
CREATE OR REPLACE TABLE DECISIONES_CREDITO AS
SELECT
    solicitud_id,
    dti_ratio,
    pago_medio_historico,
    total_impagos_30d,
    importe_solicitado,
    es_moroso AS moroso_real,
    scoring_credito!PREDICT(
        OBJECT_CONSTRUCT(
            'EDAD', edad, 'DTI_RATIO', dti_ratio,
            'INGRESOS_MENSUALES', ingresos_mensuales,
            'GASTOS_MENSUALES', gastos_mensuales,
            'ESTABILIDAD_EMPLEO', estabilidad_empleo,
            'EMPLEO_ESTABLE', empleo_estable,
            'NUM_DEPENDIENTES', num_dependientes,
            'ES_PROPIETARIO', es_propietario,
            'ANTIGUEDAD_BANCARIA', antiguedad_bancaria,
            'NUM_CREDITOS_PREVIOS', num_creditos_previos,
            'PAGO_MEDIO_HISTORICO', pago_medio_historico,
            'TOTAL_IMPAGOS_30D', total_impagos_30d,
            'TOTAL_IMPAGOS_90D', total_impagos_90d,
            'IMPORTE_SOLICITADO', importe_solicitado,
            'PLAZO_MESES', plazo_meses
        )
    ) AS prediccion,
    ROUND(prediccion['probability']['true']::FLOAT * 100, 1) AS prob_moroso_pct,
    CASE
        WHEN prediccion['probability']['true']::FLOAT >= 0.6 THEN 'RECHAZADO'
        WHEN prediccion['probability']['true']::FLOAT >= 0.3 THEN 'CONDICIONADO'
        ELSE 'APROBADO'
    END AS decision
FROM TEST_CREDITO;

SELECT decision, COUNT(*) AS n, ROUND(AVG(prob_moroso_pct),1) AS prob_media
FROM DECISIONES_CREDITO GROUP BY 1 ORDER BY 2 DESC;

---

## Paso 7: Explicabilidad con IA

### ¿Por Qué Explicar Decisiones?

La regulación bancaria (EBA, GDPR) exige que las decisiones crediticias sean **transparentes y explicables**.

### `CORTEX.COMPLETE()` para Explicaciones

Usamos el LLM de Snowflake para generar explicaciones en lenguaje natural de cada decisión, basadas en los factores de riesgo.

In [ ]:
-- Generar explicaciones IA para las decisiones
CREATE OR REPLACE TABLE EXPLICACIONES_CREDITO AS
SELECT
    solicitud_id,
    decision,
    prob_moroso_pct,
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large',
        'Eres un analista de riesgo crediticio bancario. Genera una explicación breve (3 líneas) de esta decisión crediticia:\n' ||
        'Decisión: ' || decision || '\n' ||
        'Probabilidad de impago: ' || prob_moroso_pct || '%\n' ||
        'DTI ratio: ' || ROUND(dti_ratio, 2) || '\n' ||
        'Historial de pagos: ' || ROUND(pago_medio_historico, 1) || '%\n' ||
        'Impagos previos: ' || total_impagos_30d || '\n' ||
        'Importe solicitado: ' || importe_solicitado || '€\n' ||
        'Responde solo con la explicación en español.'
    ) AS explicacion_ia
FROM DECISIONES_CREDITO
SAMPLE (50 ROWS);

SELECT solicitud_id, decision, prob_moroso_pct, explicacion_ia 
FROM EXPLICACIONES_CREDITO LIMIT 5;

---

## Paso 8: Dashboard de Decisiones

### Dashboard Interactivo

Muestra KPIs de decisiones crediticias, distribución por resultado y tabla de solicitudes con explicación IA.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Scoring de Riesgo de Crédito')
st.markdown('### Panel de Decisiones Crediticias')

total = session.sql('SELECT COUNT(*) FROM DECISIONES_CREDITO').collect()[0][0]
aprobados = session.sql("SELECT COUNT(*) FROM DECISIONES_CREDITO WHERE decision='APROBADO'").collect()[0][0]
rechazados = session.sql("SELECT COUNT(*) FROM DECISIONES_CREDITO WHERE decision='RECHAZADO'").collect()[0][0]

c1, c2, c3, c4 = st.columns(4)
c1.metric('Total Solicitudes', f'{total:,}')
c2.metric('Aprobadas', f'{aprobados:,}')
c3.metric('Rechazadas', f'{rechazados:,}')
c4.metric('Tasa Aprobación', f'{aprobados/total*100:.1f}%')

st.markdown('---')
st.subheader('Distribución de Decisiones')
df = session.sql("SELECT decision, COUNT(*) AS n FROM DECISIONES_CREDITO GROUP BY 1").to_pandas()
st.bar_chart(df.set_index('DECISION'))

st.markdown('---')
st.subheader('Solicitudes con Explicación IA')
df_exp = session.sql('SELECT solicitud_id, decision, prob_moroso_pct, explicacion_ia FROM EXPLICACIONES_CREDITO ORDER BY prob_moroso_pct DESC LIMIT 20').to_pandas()
st.dataframe(df_exp)

---

## Paso 9: Limpieza del Entorno (Opcional)

**Esta celda está comentada por defecto.**

⚠️ **Advertencia**: Eliminará todos los datos y modelos.

In [ ]:
-- Descomentar las líneas siguientes para limpiar el entorno

-- DROP DATABASE IF EXISTS SCORING_CREDITO_DB;